In [ ]:
!pip install ultralytics onnxruntime-gpu tensorflow -q


# Edge Optimization

This notebook evaluates the trained YOLOv8s navigation model under different deployment formats and quantization levels.

The final model is exported from PyTorch FP32 to ONNX FP16, ONNX INT8, and TFLite INT8, followed by model-size and validation comparisons.

In [ ]:
import torch
import cv2
import time
import shutil
import os
from PIL import Image
import glob
import numpy as np
from pathlib import Path
from ultralytics import YOLO
import random
import yaml
import matplotlib.pyplot as plt
import onnxruntime as ort
from tensorflow.lite.python.interpreter import Interpreter
from collections import Counter


## Dataset and Model Setup

The trained three-class model and validation dataset are loaded for the edge optimization experiments.

In [ ]:
data_root = Path("/kaggle/input/datasets/tarunpandianm/eric-dataset/dataset")

for split in ["train", "val", "test"]:
    img_dir = data_root / split / "images"
    lbl_dir = data_root / split / "labels"

    img_count = len(list(img_dir.glob("*")))
    lbl_count = len(list(lbl_dir.glob("*.txt")))

    class_counts = Counter()
    for f in lbl_dir.glob("*.txt"):
        for line in f.read_text().splitlines():
            if line.strip():
                class_counts[int(line.split()[0])] += 1

    print(f"{split}:")
    print(f"  images: {img_count} | labels: {lbl_count}")
    print(f"  barrier(0): {class_counts[0]}  cone(1): {class_counts[1]}  stop_sign(2): {class_counts[2]}")
    print()


In [ ]:
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
cpu_count = os.cpu_count()
print(f"CPU cores: {cpu_count}")

In [ ]:
model_path = "/kaggle/input/datasets/tarunpandianm/eric-dataset/eric_v3_best.pt"
val_dir = Path("/kaggle/input/datasets/tarunpandianm/eric-dataset/dataset/val/images")
img_path = list(val_dir.glob('*'))[:100]

In [ ]:
def benchmark_pt(model_path, device, img_paths, runs=100):
    """
    Full pipeline benchmark:
    imread -> preprocess -> forward -> postprocess -> total
    """
    model = YOLO(model_path)
    model.to(device)

    for p in img_paths[:5]:
        model.predict(source=str(p), device=device, verbose=False)

    times = []
    for p in img_paths:
        t0 = time.perf_counter()

        # full pipeline — read + preprocess + forward + postprocess
        img     = cv2.imread(str(p))
        results = model.predict(
            source  = img,
            device  = device,
            verbose = False
        )

        t1 = time.perf_counter()
        times.append((t1 - t0) * 1000)  # ms

    avg_ms = np.mean(times)
    std_ms = np.std(times)
    fps    = 1000 / avg_ms

    return {
        "avg_ms" : round(avg_ms, 2),
        "std_ms" : round(std_ms, 2),
        "fps"    : round(fps, 1),
        "min_ms" : round(min(times), 2),
        "max_ms" : round(max(times), 2),
    }

## Model Export and Quantization

The trained FP32 PyTorch model is exported to FP16 ONNX, INT8 ONNX, and INT8 TFLite formats to compare deployment-oriented model representations.

In [ ]:
#fp32 CPU
print("FP32 CPU")
fp32_cpu = benchmark_pt(model_path, 'cpu', img_path)
print(f"  avg: {fp32_cpu['avg_ms']}ms | fps: {fp32_cpu['fps']} |"
      f"std: {fp32_cpu['std_ms']}ms")

In [ ]:
#fp32 GPU
print("FP32 GPU")
fp32_gpu = benchmark_pt(model_path, 'cuda:0', img_path)
print(f"  avg: {fp32_gpu['avg_ms']}ms | fps: {fp32_gpu['fps']} |"
      f"std: {fp32_gpu['std_ms']}ms")

In [ ]:
WORK_PATH  = "/kaggle/working/best_v3.pt"
shutil.copy(model_path, WORK_PATH)
model = YOLO(WORK_PATH)

In [ ]:
fp32_mb = os.path.getsize(WORK_PATH) / (1024 ** 2)

#fp16 onnx
model.export(format = 'onnx',half = True, imgsz= 640)
FP16_PATH = WORK_PATH.replace(".pt", ".onnx")
fp16_mb   = os.path.getsize(FP16_PATH) / (1024**2)
print(f"  saved: {fp16_mb:.1f} MB")
os.rename("/kaggle/working/best_v3.onnx", "/kaggle/working/best_v3_fp16.onnx")

#int8 onnx
model.export(format="onnx", int8=True, imgsz=640)
INT8_PATH = WORK_PATH.replace(".pt", "_int8.onnx")
fp8_mb    = os.path.getsize(INT8_PATH) / (1024**2)
print(f"  saved: {fp8_mb:.1f} MB")

print(f"\nFP32: {fp32_mb:.1f} MB")
print(f"FP16: {fp16_mb:.1f} MB  ({fp32_mb/fp16_mb:.1f}x smaller)")
print(f"INT8: {fp8_mb:.1f} MB   ({fp32_mb/fp8_mb:.1f}x smaller)")

In [ ]:
model.export(format = 'tflite', int8 = True, imgsz = 640)
tflite_files = glob.glob("/kaggle/working/**/*.tflite", recursive=True)
print(f"TFLite files found: {tflite_files}")

if tflite_files:
    TFLITE_PATH = tflite_files[0]
    tflite_mb   = os.path.getsize(TFLITE_PATH) / (1024**2)
    print(f"TFLite INT8 size: {tflite_mb:.1f} MB  ({fp8_mb/tflite_mb:.1f}x smaller)")

In [ ]:
import yaml


data = {
    "train": f"{data_root}/train/images",
    "val":   f"{data_root}/val/images",
    "test":  f"{data_root}/test/images",
    "nc":    3,
    "names": ["barrier", "cone", "stop_sign"]
}

FIXED_YAML = "/kaggle/working/data_fixed.yaml"
with open(FIXED_YAML, "w") as f:
    yaml.dump(data, f, default_flow_style=False, sort_keys=False)

print(open(FIXED_YAML).read())

In [ ]:
def show_confusion_matrix(model_path, data_yaml=FIXED_YAML):
    file_ext = Path(model_path).suffix.lower()
    run_name = Path(model_path).stem + "_" + file_ext.replace('.', '')
    model = YOLO(model_path)
    device_target = 'cpu' if file_ext in ['.onnx', '.tflite'] else 0

    metrics = model.val(
        data=data_yaml,
        plots=True,
        name=run_name,
        device=device_target
    )

    print(f"\n{run_name.upper()} RESULTS")
    print(f"Model mAP50: {metrics.box.map50:.4f}")

    matrix_path = os.path.join(metrics.save_dir, 'confusion_matrix.png')
    if os.path.exists(matrix_path):
        display(Image.open(matrix_path))

In [ ]:
with open("/kaggle/input/datasets/tarunpandianm/eric-dataset/dataset/data.yaml", 'r') as f:
    print(f.read())


## Model Evaluation

A common validation function is used to evaluate the exported models and generate their confusion matrices.

In [ ]:
show_confusion_matrix("/kaggle/working/best_v3.pt")
show_confusion_matrix("/kaggle/working/best_v3_fp16.onnx")
show_confusion_matrix("/kaggle/working/best_v3_int8.onnx")
show_confusion_matrix("/kaggle/working/best_v3_int8.tflite")


## Visual Comparison

The same five validation images are passed through each model format to visually compare their detection outputs.

In [ ]:
val_dir   = Path(data_root) / "val" / "images"
all_val_imgs = list(val_dir.glob("*"))

random.seed(4)  # fixed seed so same 5 images every run
SAMPLE_IMGS = random.sample(all_val_imgs, 5)

MODEL_PATHS = {
    "FP32 PT"     : "/kaggle/working/best_v3.pt",
    "FP16 ONNX"   : "/kaggle/working/best_v3_fp16.onnx",
    "INT8 ONNX"   : "/kaggle/working/best_v3_int8.onnx",
    "INT8 TFLite" : "/kaggle/working/best_v3_int8.tflite",
}

CLASS_NAMES = {0: "barrier", 1: "cone", 2: "stop_sign"}
COLORS      = {0: (0,0,255), 1: (0,165,255), 2: (0,255,0)}

full_val_speed = {}

# ── for each model: show 5-image grid + benchmark full val folder ──
for name, path in MODEL_PATHS.items():
    print(f"{name}")
    print()

    file_ext = Path(path).suffix.lower()
    device   = "cpu" if file_ext in [".onnx", ".tflite"] else 0

    model = YOLO(path)

    # ── grid of 5 fixed images ──────────────────────────────────
    fig, axes = plt.subplots(1, 5, figsize=(25, 5))
    fig.suptitle(name, fontsize=16)

    for ax, img_path in zip(axes, SAMPLE_IMGS):
        img     = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        results = model.predict(source=str(img_path), conf=0.3,
                                 device=device, verbose=False)
        boxes = results[0].boxes
        annotated = img_rgb.copy()

        if boxes is not None:
            for box in boxes:
                cls_id = int(box.cls[0])
                x1,y1,x2,y2 = map(int, box.xyxy[0].tolist())
                color  = COLORS.get(cls_id, (255,255,255))
                cv2.rectangle(annotated, (x1,y1), (x2,y2), color, 2)
                cv2.putText(annotated, CLASS_NAMES.get(cls_id,"?"),
                            (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX,
                            0.5, color, 2)

        ax.imshow(annotated)
        ax.axis("off")

    plt.tight_layout()
    plt.savefig(f"/kaggle/working/grid_{name.replace(' ','_')}.png",
                dpi=120, bbox_inches="tight")
    plt.show()

print("FULL VAL FOLDER SPEED SUMMARY")
print(f"{'Format':<14} {'Images':>8} {'Avg(ms)':>10} {'FPS':>8}")
for name, stats in full_val_speed.items():
    print(f"{name:<14} {stats['n_imgs']:>8} {stats['avg_ms']:>10} {stats['fps']:>8}")